In [1]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 

from utils import functions as func
from tqdm import tqdm
from utils import parameters as params
import copy
from utils import metabolites as metab
import pickle
import sympy

import numpy as np
import pandas as pd
import pickle

lp_path = '/data2/hratch/human_me/other/test_lp/'

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


In [3]:
jabba = True

counter = 1
base = 0

mu_val = 1e-9
fn = '/data2/hratch/human_me/other/test_lp/S_matrix.h5'

In [3]:
from expression import build_me_model
tme, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                                unmodeled_protein_frac = None,
                                                model_id = 'toy_me_model')

# if jabba:
#     for r in tme.reactions:
#         if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
#             r._lower_bound = -1000
#             r._upper_bound = 1000
    
    
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

# with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

Generate ubiquitin reactions for proteasomal degradation
Generate ribosome


  1%|          | 7/591 [00:00<00:08, 66.23it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:10<00:00, 54.00it/s]


Generate protein expression reactions for expression module enzymes, this step may take a few minutes


...

Check reaction mass balances


 11%|█▏        | 1428/12605 [00:00<00:00, 14243.10it/s]

Check correct coupling of metabolic machinery


100%|██████████| 12605/12605 [00:06<00:00, 1862.55it/s]

Time to build: 4.702466166019439 minutes


In [4]:
sln, stat, _ = tme.solve_lp(mu_val = mu_val)

S_1 = tme.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
res = pd.DataFrame(data = {'reaction_fluxes': sln[:len(tme.reactions)]})
res.index = [r.id for r in tme.reactions]

if stat == 0:
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
else:
    infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)
    print('Model did not solve')
    
res.loc[[i for i in res.index if 'biomass' in i],:]

../../../scripts/core/model.py:261 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 239.227 seconds with status 1
Model did not solve


,reaction_fluxes
biomass_dilution,1.000000e-09
DNA_biomass_to_biomass,1.400000e-11
carbohydrate_biomass_to_biomass,7.100000e-11
lipid_biomass_to_biomass,9.700000e-11
tRNA_biomass_to_biomass,5.961321e-22
rRNA_biomass_to_biomass,1.217194e-12
mRNA_biomass_to_biomass,2.984510e-18
premRNA_biomass_to_biomass,0.000000e+00
other_rna_biomass_to_biomass,-8.740105e-22
DNA_biomass_formation,1.000000e-09


In [5]:
def save_me_model(me_model, counter):
    print('Success, please update git')
    lp_path = '/data2/hratch/human_me/test_lp/'
    me_model.pickle(lp_path + 'working_version_' + str(counter) + '.pickle')

def get_changes(S_1, S_0):
    mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
    am = S_1.index.tolist()
    mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm:
        mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

    am, rm = S_1.index.tolist(), S_1.columns.tolist()
    mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm_2:
        mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))
    
    return mismatch, mm, mm_2

In [5]:
S_0 = pd.read_hdf(fn, key = str(base))
S_1 = pd.read_hdf(fn, key = str(counter))



In [6]:
# S_0 = pd.read_hdf(fn, key = str(base))

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
            indeces = False
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch, mm, mm_2 = get_changes(S_1, S_0)
            print('Dataframes are not equal due to stoichiometric values mismatch, will not save model')
        else:
            save_me_model(tme, counter)
    else:
        print('Dataframes are not equal due to column/row label mismatch, will not save model')
else:
    save_me_model(tme, counter)

Dimensions are not the same
Columns are not the same
Rows are not the same
Dataframes are not equal due to column/row label mismatch, will not save model


In [10]:
set(S_1.columns.tolist()).difference(S_0.columns.tolist())

{'HGNC:10766_TRANSCRIPTION_PROCESSING',
 'HGNC:10325_TRANSCRIPTION_PROCESSING',
 'HGNC:9557_TRANSCRIPTION_ELONGATION',
 'HGNC:9262_mRNA_EXPORTtn',
 'HGNC:10419_TRANSCRIPTION_PROCESSING',
 'HGNC:10397_mRNA_EXPORTtn',
 'HGNC:5232_mRNA_EXPORTtn',
 'HGNC:17325_TRANSCRIPTION_PROCESSING',
 'HGNC:9530_mRNA_EXPORTtn',
 'HGNC:16696_mRNA_EXPORTtn',
 'HGNC:4801_mRNA_EXPORTtn',
 'HGNC:9558_mRNA_EXPORTtn',
 'HGNC:10933_TRANSCRIPTION_PROCESSING',
 'HGNC:17310_TRANSCRIPTION_PROCESSING',
 'HGNC:2483_TRANSCRIPTION_ELONGATION',
 'HGNC:4878_TRANSCRIPTION_PROCESSING',
 'HGNC:801_mRNA_EXPORTtn',
 'HGNC:29938_TRANSCRIPTION_PROCESSING',
 'HGNC:10354_TRANSCRIPTION_PROCESSING',
 'HGNC:23400_mRNA_EXPORTtn',
 'HGNC:11004_TRANSCRIPTION_PROCESSING',
 'HGNC:846_TRANSCRIPTION_PROCESSING',
 'HGNC:24624_TRANSCRIPTION_ELONGATION',
 'HGNC:24671_TRANSCRIPTION_ELONGATION',
 'HGNC:11159_TRANSCRIPTION_PROCESSING',
 'HGNC:2265_TRANSCRIPTION_PROCESSING',
 'HGNC:30242_TRANSCRIPTION_PROCESSING',
 'HGNC:17688_mRNA_EXPORTtn',
 'H

In [9]:
S_0.shape

(8068, 10526)

In [27]:
mismatch, mm, mm_2 = get_changes(S_1, S_0)

In [ ]:
# set(S_1.index).difference(S_0.index)
# set(S_0.index).difference(S_1.index)
# S_1.index = pd.Series(S_1.index).replace(to_replace = '290067835_complex_n', value = '3016239816_complex_n', 
#                                         inplace = False)

In [31]:
list(mm.keys())[:10]

[557, 2031, 2041, 2049, 2057, 2065, 2073, 2081, 2089, 2097]

In [52]:
m_idx = 2041
mm_2[m_idx]

{'id': 'HGNC:10420_mrna_c', 'reactions': ['TRANSLATION_ELONGATIONc']}

In [53]:
mm[m_idx]

{'id': 'HGNC:10420_mrna_c', 'reactions': [1309]}

In [54]:
S_1.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:10420_TRANSLATION_ELONGATIONc   -0.000004
Name: HGNC:10420_mrna_c, dtype: float64

In [55]:
S_0.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:10420_TRANSLATION_ELONGATIONc   -0.000004
Name: HGNC:10420_mrna_c, dtype: float64

In [57]:
test = S_1.iloc[m_idx, mm[m_idx]['reactions']]  - S_0.iloc[m_idx, mm[m_idx]['reactions']]
test[abs(test) > 1e-8]

HGNC:10420_TRANSLATION_ELONGATIONc    2.467319e-07
Name: HGNC:10420_mrna_c, dtype: float64

In [191]:
# res0 = pd.read_csv(lp_path + 'works_trash.csv', index_col = 0)
# res['og_fluxes'] = res0.loc[res.index.tolist(), :]['reaction_fluxes'].tolist()
# res['diff'] = res['reaction_fluxes'] - res['og_fluxes']

# testing ubiquitin cleavage

In [ ]:
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    
